# CUDA Vector Search — Colab GPU Diagnostics

Single-GPU telemetry + batch sweep + controlled stress cases.

Does **not** overwrite Phase 4/5 historical CSVs.

Docs: `docs/COLAB_DIAGNOSTICS.md`

## 0. Runtime

Colab: **Runtime → Change runtime type → GPU** (T4 is fine).

In [ ]:
!nvidia-smi || echo "No GPU — enable Colab GPU runtime"

In [ ]:
import os
if not os.path.exists("CMakeLists.txt"):
    !git clone https://github.com/amberfxy/vector-search-cuda.git
    %cd vector-search-cuda
else:
    print("already in repo root")
!pwd

In [ ]:
!sudo apt-get update -qq && sudo apt-get install -y -qq cmake g++ libomp-dev
!nvcc --version | head -5
!pip install -q pynvml

In [ ]:
!rm -rf build
!cmake -B build -DCMAKE_BUILD_TYPE=Release \
  -DCMAKE_CUDA_COMPILER=$(which nvcc) \
  -DCMAKE_CUDA_ARCHITECTURES=75
!cmake --build build -j 2 --target bench_batch bench_harness diagnostics_stress
!ls -l build/bench_batch build/diagnostics_stress

In [ ]:
!./build/bench_batch --num-vectors 10000 --dim 384 --batch 8 --warmup 1 --iterations 3 --method tiled || true

In [ ]:
!python3 scripts/colab_diagnostics.py \
  --num-vectors 100000 --dim 384 \
  --batches 1,8,32,128 --method tiled \
  --warmup 3 --iterations 10

In [ ]:
from pathlib import Path
p = Path("results/diagnostics/latest/REPORT.md")
print(p.read_text() if p.exists() else "REPORT.md missing")

## Notes

- Historical verified T4 metrics stay in `results/phase1_benchmark.csv` and `docs/PHASE5_WARP_PER_VECTOR.md`.
- This notebook only records **this Colab session's** telemetry + sweeps.
- Scope: single GPU; no NVLink / multi-GPU claims.